In [44]:
import random, time, tkinter as tk
# random -> مكتبة بتخليك تختار عشوائي سواء كانت من ليستاية او رقم واستعملناها في اللابات
# time -> بتحسب الوقت اللي بيعدي هنحتاجها في الواجهة 
# tinker -> دي مكتبة بتاعت بايثون استخدمناها عشان نعمل الواجهة الاساسية لو محدش غيرها
N, TIME_LIMIT, NUM_WORDS, MAX_EFFORT = 8, 300, 10, 1000
# N -> grid size aka 8*8 , TIME_LIMIT -> literally time limit in seconds
# NUM_WORDS ->  عدد الكلام المستعمل في اللعبة انا شايف انو 10 احسن حاجة عشان هتوضح ليه بنفضل ستايلات على ستايلات تانية 
# MAX_EFFORT -> اعلى عدد من محاولات انك تحط كلمة في اماكن مختلفة 
POOL = ["HOUSE", "WATER", "APPLE", "TABLE", "CHAIR", "MUSIC", "LIGHT", "PHONE", "BREAD",
        "GREEN", "SMILE", "RIVER", "CLOUD", "PLANT", "TRAIN", "SUGAR", "MONEY", "HAPPY",
        "SUMMER", "WINTER", "GARDEN", "FRIEND", "FAMILY", "SCHOOL", "WINDOW", "DOCTOR",
        "ORANGE", "KITCHEN", "MORNING", "COUNTRY", "BIRTHDAY", "MOUNTAIN", "ANIMAL"]
# هنا زود كلام براحتك في الليستاية دي اهم حاجة الكلمة متعديش ال 8 حروف
DIRS = [(0, 1), (1, 0), (1, 1), (-1, 1), (0, -1), (-1, 0), (-1, -1), (1, -1)]
# دي الاتجهات هنحتاجها في المعادلة تحت على طول

def placements(grid, word):
# هنا الفنكشن دي بتتاكد من كل الاماكن اللي كل حرف من الكلمة ينفع يتحط فيها لو المكان فيه نفس الحرف او فاضي هيعدي غير كدا هنروح للي بعدها
    out = []
    for r in range(N):
        for c in range(N):
            for dr, dc in DIRS:
# السطر اللي تحت ده من الاخر بيعمل ليستاية جواها كل الاماكن اللي الكلمة ينفع تتحط فيها بيجرب كل الاتجاهات وكل عمود في كل صف 
# CAT مثال على كلمة 
# cat is 3 letters so index 0 1 2 , cells is a list which will have three tuples of placements for each letter of the word cat 
#let's say the index row in the loop is 3 and column is 2 and the direction is (1,-1) the loop underneath will make the cells = [(3,2),(4,1),(5,0)]
#which if you look at the grid is a diagonal from right to left      
                cells = [(r + dr * i, c + dc * i) for i in range(len(word))]
                if all(0 <= y < N and 0 <= x < N and grid[y * N + x] in ".%s" % ch
                       for (y, x), ch in zip(cells, word)):
# all makes sure that every condition is true
# first zip(cells,word) makes it so that every placement is with it's character in the word like this ((3, 2), 'C'), ((4, 1), 'A'), ((5, 0), 'T') 
# so the if condition only makes sure that everything is true as in y "row index" is bigger or equal to 0 and smaller than the number of grid which is 8 so maximum index is 7
# column is same as row and that the cell number since our grid is literally a big string of N*N we calculate the index number using y * N + x we make sure that it's either a . which is empty or the character that we are iterating through and if allat is True we append the cells to out
                    out.append(cells)
    return out

# هنا الفنكشن دي حرفيا بتحط الحروف في مكانها في الجريد
def place(grid, word, cells):
#  بتاخد كوبي من الجريد وبتحطها في ليست عشان تعدل عليها  
    g = list(grid)
    for (y, x), ch in zip(cells, word):
        g[y * N + x] = ch
# هنا ال ريتيرن بترجع الليستاية كا سترينج تاني بعد ما اتعدل عليها
    return "".join(g)


# دي التلت ستايلات اللي عندنا كل واحدة بتاخد الجريد والسيلز وبتشتغل عليهم 
STYLES = {
    # overlap 
    # sort كل مكان في السيلز عبارة عن مكان على الجريد الفنكشن بترجع مجموع الاماكن اللي لقيتها مش فاضية وبتحط في الاخر سالب عشان فنشن ال 
    # تعتبرها اختيار افضل
    "overlap": lambda grid, cells: -sum(grid[y * N + x] != "." for y, x in cells),
    # بالنسبة لل edge
    #   هنا بتفضل لما تبقى اماكن السيلز بعيدة عن النص بتاع الجريد مثال لو حسبت مجموع السيلز دي
    # (0,0) , (0,1) , (0,2)
    # outcome : 18 and with the outside negative it becomes -18 وزي ما قولت السالب بيخليها افضل لما نستعمل عليها فنكشن السورت
    "edge":    lambda grid, cells: -sum(abs(y - (N - 1) / 2) + abs(x - (N - 1) / 2) for y, x in cells),
    # هنا حرفيا عشوائية وكل ما كان الرقم اصغر كل ما فنكشن السورت اعتمدته الاول
    "random":  lambda grid, cells: random.random(),
}
# هنا حرفيا ديكشنري لكل ستايل جواه ليست فاضية هتتملي بعدين بعدد المحاولات الستايل احتاجها عشان يعمل لعبة لما نستعملهم عشان نعرف اني واحدة اسهل واحسن 
history = {s: [] for s in STYLES} 


def pick_style():
#هنا هيشوفهم واحد واحد قبل اي حاجة هيشوف لو الستايل في الديكشنري عبارة عن ليستاية فاضية هيجربه عشان مجربهوش قبل كدا
    for s in STYLES:
        if not history[s]:
            return s
#    فيه نسبة انو يجرب ستايل عشوائي لو الرقم العشوائي طلع اقل من 0.2 او نسبة 20% يعني
    if random.random() < 0.2:                      
        return random.choice(list(STYLES))
#  لو عدا من كل حاجة من دي في العادي بيختار احسن ستايل بانه بيروح لكل واحد ويحسب عدد محاولاته على اشتغل كام مرة واقل رقم هيطلع احسن واحد فا هيختاره
# example : history = {
#    "overlap": [100, 200, 300],
#    "edge": [50, 60, 70],
#    "random": [400, 500, 600]
#}     
# (100 + 200 + 300) / 3 = 200 , (50 + 60 + 70) / 3 = 60 , (400 + 500 + 600) / 3 = 500
# edge اقل رقم ال
    return min(STYLES, key=lambda s: sum(history[s]) / len(history[s]))

# exception هنحتاج ال   
# عشان احنا في ال فنكشن اللي بعدها باكتراك هنقعد ننادي الفنكشن جوا نفسها كتير فا لما عدد المحاولات يعدي الماكس بتاعنا اللي هو كان 1000 عايزين نطلع من كل الفنكشنز اللي جوا بعض دي على طول
class TooHard(Exception):
    pass

# دي زي وكانك عملت كل واحدة لوحدها عادي هي هي
effort = backtracks = 0

# هنا بقى واحدة واحدة انا هشرحها كاملة من فوق هنا عشان مينفعش ادخل جواها هيبقى شكلها معقد
# grid -> N*N string of "." , words -> list of randomly selected words , i which is index which is 0 , style -> chosen style
# uses the same global variables that are outside the function
# if i == len(words) that means the function recurred till the end with no issues 
# we get this possible placements for the current grid and the current word inside a variable named options and we shuffle them so it lowers the chance of getting the same puzzle twice
# and then we sort them using the selected style and yes even if we sort them we still need the shuffle since if two placements have the same value the sort is gonna leave them as their original order
# again options are like this options = [  [(0, 0), (0, 1), (0, 2)], [(2, 1), (2, 2), (2, 3)], [(4, 3), (4, 4), (4, 5)]  ] so cells in options are a list of tuples 
# now for every time we try a cell the effort += 1 so we can measure if it's taking too many attempts and it's not worth it and if we go over the max effort we raise the exception toohard which interrupts every function in the recursion and stops them
# now for the bigger picture now let's say the first word is DOG the second is OWL now we go through every cell in options for DOG now let's say we go through the first one now the result awaits for another call of the function backtrack 
# but this time it's a new grid which is a copy of the original modified with the placement that we used for DOG and i + 1 so it goes to the next word which is OWL now again we see all the possible options for this word with the new grid which DOG is placed in already
# now we try the same thing with OWL we go through all of it's possible cells now let's say we go through the first one the same cycle repeats with the next words but if the next word doesn't have a valid placement we go back one cycle now let's say we chose a placement for OWL and then the next cycle was FOX but FOX doesn't have any valid placement 
# then it returns None for the previous cycle which was OWL now owl's loop tries another placement and i think you get the idea by now now let's say it all worked out and the final word found a valid placement the new word placement is placed in the new copied grid it made and the first condition becomes True which then returns the final grid of all placed words the final grid keeps returning to the previous cycles till it returns to the very first one and returns the final grid
# now again if the effort exceeds max effort we raise the too hard exception and all of this recursion is interrupted and it doesn't make any problems since every cycle edits it's own copied grid
# also everytime a cycle returns None to it's previous cycle and it makes it's loop try another placement (cells in options) we add + 1 to the backtracks 
def backtrack(grid, words, i, style):
    global effort, backtracks
    if i == len(words):
        return grid
    options = placements(grid, words[i])
    random.shuffle(options)
    options.sort(key=lambda cells: STYLES[style](grid, cells))
    for cells in options:
        effort += 1
        if effort > MAX_EFFORT:
            raise TooHard                         
        result = backtrack(place(grid, words[i], cells), words, i + 1, style)
        if result:
            return result
        backtracks += 1                            
    return None


def make_puzzle():
    # uses the same global effort , backtracks variables
    global effort, backtracks
    while True:
    # using the pick_style function to pick a style
        style = pick_style()
    # picking random different words from the pool of words how many is indicated by the NUM_WORDS variable
        words = random.sample(POOL, NUM_WORDS)
    # long words first = fewer dead ends   
        words.sort(key=len, reverse=True)  
    # zeroing everything every loop
        effort = backtracks = 0
        try:
    # here we try the backtrack function to make the grid of our game
            grid = backtrack("." * (N * N), words, 0, style)
        except TooHard:
    # and if we ever raise the toohard exception we catch it here and append to the list of the of the style key in the dictionary and we continue another round of the loop so in summary we try again
            history[style].append(MAX_EFFORT)       # punish the style, then try again
            continue
    # here if we actually succeed and get our final grid with all the valid placements 
        if grid:
    # we append the number of efforts it took to the list of the of the style key in the dictionary so that we can check if this is the most efficient when making another puzzle by it taking fewer attempts on average
            history[style].append(effort)   
            # here we just take the final grid and if there is an empty position which is indicated by "." we replace it with a random character and if not we leave the original character
            grid = "".join(random.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") if ch == "." else ch
                           for ch in grid)
            return words, grid, style, effort, backtracks


# ---------------- GAME / GUI PART ----------------
root = tk.Tk()
root.title("AI Word Search")
state = dict(words=[], grid="", found=set(), done=set(), first=None, preview=[],
             left=0, score=0, running=False, job=None)

tk.Button(root, text="New Game", command=lambda: new_game()).pack(pady=5)
info = tk.Label(root, font=("Arial", 12)); info.pack()
msg = tk.Label(root, font=("Arial", 12, "bold")); msg.pack()
board = tk.Frame(root); board.pack(padx=10, pady=5)
wordlabel = tk.Label(root, font=("Arial", 12)); wordlabel.pack()
note = tk.Label(root, font=("Arial", 11), fg="red"); note.pack()
statlabel = tk.Label(root, font=("Arial", 9), fg="gray"); statlabel.pack(pady=5)

labels = []


def build_board():
    for l in labels:
        l.destroy()
    labels.clear()
    for r in range(N):
        for c in range(N):
            l = tk.Label(board, width=3, height=1, font=("Arial", 18, "bold"), relief="ridge")
            l.grid(row=r, column=c)
            l.bind("<ButtonPress-1>", lambda e, r=r, c=c: press(r, c))
            l.bind("<B1-Motion>", lambda e: motion(e))
            l.bind("<ButtonRelease-1>", lambda e: release(e))
            labels.append(l)


def refresh():
    for k, l in enumerate(labels):
        color = "white"
        if (k // N, k % N) in state["found"]: color = "lightgreen"
        if (k // N, k % N) in state["preview"]: color = "orange"   # live drag highlight
        if state["first"] == (k // N, k % N): color = "yellow"
        l.config(text=state["grid"][k], bg=color)
    wordlabel.config(text="Words found: %d / %d" % (len(state["done"]), len(state["words"])))
    info.config(text="Time: %d   Score: %d" % (state["left"], state["score"]))


def new_game():
    if state["job"]:
        root.after_cancel(state["job"])
    note.config(text="")
    t = time.time()
    words, grid, style, tries, undone = make_puzzle()
    state.update(words=words, grid=grid, found=set(), done=set(), first=None,
                 left=TIME_LIMIT, running=True)
    msg.config(text="AI used '%s': %d attempts, %d backtracks, %.2fs"
               % (style, tries, undone, time.time() - t), fg="black")
    statlabel.config(text="AI's avg attempts (lower = better):  " + "   ".join(
        "%s %.0f" % (s, sum(v) / len(v)) for s, v in history.items() if v))
    refresh()
    tick()


def tick():
    if not state["running"]:
        return
    if state["left"] <= 0:
        state["running"] = False
        msg.config(text="TIME'S UP - YOU LOSE", fg="red")
        return
    state["left"] -= 1
    refresh()
    state["job"] = root.after(1000, tick)


def cell_at(e):
    """Row, col of the cell under the mouse (or None if outside the board)."""
    for k, l in enumerate(labels):       # ask each cell where it really is on screen
        x, y = l.winfo_rootx(), l.winfo_rooty()
        if x <= e.x_root < x + l.winfo_width() and y <= e.y_root < y + l.winfo_height():
            return (k // N, k % N)
    return None


def line(start, end):
    """Cells on the straight line from start to end ([] if it isn't straight)."""
    (r0, c0), (r, c) = start, end
    dy, dx = r - r0, c - c0
    if dy != 0 and dx != 0 and abs(dy) != abs(dx):
        return []
    n = max(abs(dy), abs(dx)) + 1
    sy, sx = (dy > 0) - (dy < 0), (dx > 0) - (dx < 0)
    return [(r0 + sy * i, c0 + sx * i) for i in range(n)]


def press(r, c):
    if state["running"]:
        state["first"] = (r, c)          # drag starts here
        refresh()


def motion(e):
    if state["running"] and state["first"] is not None:
        end = cell_at(e)
        state["preview"] = line(state["first"], end) if end else []
        refresh()                        # highlights the path while you drag


def release(e):
    if state["running"] and state["first"] is not None:
        end = cell_at(e)
        if end and end != state["first"]:                 # a real drag, not a single click
            before = len(state["done"])
            check_line(line(state["first"], end))
            if len(state["done"]) == before:              # nothing new was found
                note.config(text="Not one of the hidden words")
                root.after(1500, lambda: note.config(text=""))
    state["first"], state["preview"] = None, []
    refresh()


def check_line(cells):
    text = "".join(state["grid"][y * N + x] for y, x in cells)
    for w in state["words"]:
        if w not in state["done"] and text in (w, w[::-1]):
            state["done"].add(w)
            state["found"].update(cells)
            state["score"] += 10 * len(w)
    if len(state["done"]) == len(state["words"]):
        state["running"] = False
        state["score"] += state["left"]            # time bonus
        msg.config(text="YOU WIN!", fg="green")


build_board()
new_game()
root.mainloop()